#  Hackathon — Visual RAG Local pour le Tourisme Marocain

---

```
  ╔══════════════════════════════════════════════════════════════════╗
  ║           HACKATHON IA — 3h00 — TRAVAIL INDIVIDUEL              ║
  ║                                                                  ║
  ║   C'est une course.                                             ║
  ║                                                                  ║
  ║   Le premier etudiant dont l'application Streamlit tourne       ║
  ║   et repond correctement a une photo.                     ║
  ║                                                                  ║
  ║   Pas celui avec le plus beau code.                             ║
  ║   Pas celui qui a tout compris.                                 ║
  ║   Celui dont l'app fonctionne en premier.                       ║
  ╚══════════════════════════════════════════════════════════════════╝
```

---


**Outils :** `Ollama` · `CLIP (clip-ViT-B-32)` · `ChromaDB` · `Pillow` · `Streamlit`

---

### Objectif

Construire un système capable d'**identifier un lieu touristique marocain à partir d'une photographie** et de **générer automatiquement une fiche historique** en langage naturel grâce à un LLM local — puis exposer le tout dans une **interface Streamlit opérationnelle**.

Le chronomètre tourne dès que ce notebook est ouvert. Le premier étudiant qui appelle un enseignant pour une démo live et valide les critères ci-dessous remporte le hackathon.

---




---

### Étapes conseillées

```
  [  0 min] Lire ce notebook, creer le venv, installer les dependances
  [ 15 min] Indexer vos images (Cellule 4)
  [ 30 min] Implementer la recherche (Cellule 5)
  [ 60 min] Pipeline end-to-end fonctionnel (Cellule 7)
  [ 90 min] App Streamlit en cours (Cellule 9)
  [150 min] App deployee et testee sur vos propres photos
  [180 min] DEADLINE — demo live, dernier appel
```

---


##  Partie Théorique

---

### 1. Qu'est-ce que le Visual RAG ?

Le **RAG** (*Retrieval-Augmented Generation*) est une architecture qui améliore les LLM en leur fournissant des **informations externes pertinentes** avant de générer une réponse, au lieu de se fier uniquement à leur mémoire paramétrique.

Le **Visual RAG** est une extension de ce paradigme : **le point d'entrée n'est plus un texte, mais une image.**

```
  RAG Classique (Textuel)         Visual RAG (Notre TP)
  ─────────────────────────       ──────────────────────────────
  ┌─────────────┐                 ┌─────────────────────────┐
  │ Question    │                 │   Photo d'un lieu     │
  │ (texte)     │                 │  (pixels bruts)         │
  └──────┬──────┘                 └──────────┬──────────────┘
         │ Embedding texte                   │ Embedding image (CLIP)
         ▼                                   ▼
  ┌─────────────┐                 ┌──────────────────────────┐
  │ Vector DB   │                 │  Vector DB (ChromaDB)    │
  │ (textes)    │                 │  (vecteurs d'images ref) │
  └──────┬──────┘                 └──────────┬───────────────┘
         │ Contexte récupéré                 │ Contexte : nom du lieu
         ▼                                   ▼
  ┌─────────────┐                 ┌──────────────────────────┐
  │    LLM      │                 │  LLM local (Ollama)      │
  │  (Réponse)  │                 │  → Fiche historique      │
  └─────────────┘                 └──────────────────────────┘
```

**L'enjeu fondamental :** transformer des pixels en vecteurs sémantiques comparables, permettant de retrouver des informations dans une base de connaissances par **similarité géométrique**.

---

### 2. L'Espace Latent Multimodal — Le rôle de CLIP

**CLIP** (*Contrastive Language-Image Pre-training*, OpenAI 2021) est le modèle clé de notre pipeline.  
Son innovation majeure : **projeter images ET texte dans le même espace vectoriel de haute dimension.**

```
                    ESPACE LATENT MULTIMODAL (dimension 512)
                    ─────────────────────────────────────────

   Photo de           ┌──────────────┐
   Volubilis    ──────► │  Encodeur    │──────►  [ 0.23, -0.87, 0.14, ... ]  ─┐
                        │  IMAGE (ViT) │                                        │
                        └──────────────┘                                        │  Distance
                                                                                │  cosinus ≈ 0 !
   "Ruines              ┌──────────────┐                                      │
   romaines au  ──────►  │  Encodeur    │──────►  [ 0.25, -0.84, 0.16, ... ]  ─┘
   Maroc"               │  TEXTE       │
                        └──────────────┘

  ► Un vecteur d'image et son texte descriptif sont PROCHES dans cet espace.
  ► Deux images de lieux DIFFÉRENTS ont des vecteurs ÉLOIGNÉS.
```

**Entraînement contrastif :** CLIP a été entraîné sur 400 millions de paires (image, texte) en maximisant la similarité des paires correctes et en minimisant celle des paires incorrectes.  
Résultat : il généralise à des domaines non vus, y compris nos lieux touristiques marocains.

| Modalité | Encodeur utilisé | Dimension output |
|----------|-----------------|------------------|
| Image    | Vision Transformer (ViT-B/32) | 512 |
| Texte    | Transformer modifié | 512 |

---

### 3. Le Pipeline Visual RAG — Vue d'ensemble

Notre système se décompose en **deux phases distinctes** : une phase hors-ligne (*offline*) et une phase en ligne (*online*).

```
╔══════════════════════════════════════════════════════════════════════════╗
║          PHASE 1 — INGESTION (Offline, exécutée une seule fois)         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║    dataset/                                                            ║
║   ├── tour_hassan_1.jpg  ──┐                                             ║
║   ├── tour_hassan_2.jpg  ──┤  ① Chargement                               ║
║   ├── volubilis_1.jpg    ──┤  des images                                 ║
║   ├── sefrou_1.jpg       ──┘  de référence                               ║
║          │                                                               ║
║          ▼                                                               ║
║   ┌─────────────────┐                                                    ║
║   │  CLIP ViT-B/32  │  ② Extraction des embeddings (vecteurs 512D)      ║
║   └────────┬────────┘                                                    ║
║            │                                                             ║
║            ▼                                                             ║
║   ┌─────────────────────────────────────────────┐                        ║
║   │             ChromaDB (Vector Store)         │  ③ Stockage avec       ║
║   │  id: "img_001"  vector: [0.23, ...]         │  métadonnées          ║
║   │  metadata: {name: "Tour Hassan",            │  (nom, histoire,      ║
║   │             ville: "Rabat", ...}            │   ville, coords)      ║
║   └─────────────────────────────────────────────┘                        ║
╚══════════════════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════════════════╗
║        PHASE 2 — REQUÊTE → GÉNÉRATION (Online, à chaque utilisation)    ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║    Nouvelle photo   ④ Vectorisation de la requête (CLIP)               ║
║   de l'utilisateur  ─────────────────────────────────────────────────►  ║
║                                           │                              ║
║                                           ▼                              ║
║                              ⑤ Recherche par similarité cosinus          ║
║                                  dans ChromaDB                           ║
║                                           │                              ║
║                                           ▼                              ║
║                              Contexte récupéré :                         ║
║                              {"lieu": "Tour Hassan", "histoire": "..."}  ║
║                                           │                              ║
║                                           ▼                              ║
║                              ⑥ Augmentation du prompt (contexte + query) ║
║                                           │                              ║
║                                           ▼                              ║
║                              ┌────────────────────────┐                  ║
║                              │   Ollama (LLM local)   │                  ║
║                              │   llama3 / mistral     │                  ║
║                              └────────────┬───────────┘                  ║
║                                           │                              ║
║                                           ▼                              ║
║                               "La Tour Hassan est un minaret           ║
║                                 inachevé du XIIe siècle..."              ║
╚══════════════════════════════════════════════════════════════════════════╝
```

**Résumé des 5 étapes clés :**

| # | Étape | Outil | Input → Output |
|---|-------|-------|----------------|
| ① | **Ingestion** | Pillow | Dossier images → images PIL |
| ② | **Embedding** | CLIP | Image → Vecteur 512D |
| ③ | **Stockage** | ChromaDB | Vecteurs + métadonnées → Index |
| ④⑤ | **Retrieval** | ChromaDB | Image requête → Contexte texte |
| ⑥ | **Génération** | Ollama | Contexte + Question → Réponse |

---

### 4. Structure des données attendue

Créez la structure de dossiers suivante avant de commencer :

```
visual_rag_tourisme/
│
├── dataset/
│   ├── tour_hassan/
│   │   ├── tour_hassan_1.jpg       ← vue d'ensemble
│   │   └── tour_hassan_2.jpg       ← vue rapprochée
│   │
│   ├── volubilis/
│   │   ├── volubilis_colonnes.jpg
│   │   └── volubilis_arc.jpg
│   │
│   ├── cascades_sefrou/
│   │   ├── sefrou_1.jpg
│   │   └── sefrou_2.jpg
│   │
│   ├── ait_benhaddou/
│   │   └── ait_benhaddou_1.jpg
│   │
│   └── medersa_bou_inania/
│       └── bou_inania_1.jpg
│
├── test_images/                    ← vos images de test
│   └── ma_photo_tour_hassan.jpg
│
├── metadata.json                   ← données structurées (voir ci-dessous)
├── chroma_db/                      ← généré automatiquement par ChromaDB
└── tp_visual_rag.ipynb             ← ce notebook
```

**Format du fichier `metadata.json` :**

```json
{
  "lieux": [
    {
      "id": "tour_hassan",
      "nom": "Tour Hassan",
      "ville": "Rabat",
      "coordonnees": {"lat": 34.0239, "lon": -6.8220},
      "epoque": "XIIe siècle (1195)",
      "style": "Almohade",
      "classification": "Patrimoine National",
      "histoire": "La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque commandée par le Sultan Yacoub al-Mansour en 1195. Avec ses 44 mètres, elle devait atteindre 86 mètres. La mort du Sultan en 1199 arrêta les travaux. Elle est aujourd'hui le symbole de Rabat.",
      "images": ["tour_hassan/tour_hassan_1.jpg", "tour_hassan/tour_hassan_2.jpg"]
    },
    {
      "id": "volubilis",
      "nom": "Volubilis",
      "ville": "Meknès (région)",
      "coordonnees": {"lat": 34.0728, "lon": -5.5536},
      "epoque": "IIIe siècle av. J.-C. — VIIIe siècle",
      "style": "Romain",
      "classification": "UNESCO (1997)",
      "histoire": "Volubilis est une ancienne cité berbère et romaine partiellement fouillée. Capitale du royaume de Maurétanie, puis de la province romaine de Maurétanie Tingitane, elle atteint son apogée au IIe et IIIe siècles. Inscrite au patrimoine mondial de l'UNESCO depuis 1997.",
      "images": ["volubilis/volubilis_colonnes.jpg", "volubilis/volubilis_arc.jpg"]
    },
    {
      "id": "cascades_sefrou",
      "nom": "Cascades de Sefrou",
      "ville": "Sefrou",
      "coordonnees": {"lat": 33.8307, "lon": -4.8327},
      "epoque": "Site naturel",
      "style": "Naturel",
      "classification": "Site naturel protégé",
      "histoire": "Les cascades de Sefrou, situées à 28 km de Fès, se forment grâce à l'oued Aggaï traversant les gorges de la médina. La ville de Sefrou, surnommée 'la petite Fès', est l'une des plus anciennes cités du Maroc avec une histoire remontant au IXe siècle.",
      "images": ["cascades_sefrou/sefrou_1.jpg", "cascades_sefrou/sefrou_2.jpg"]
    },
    {
      "id": "ait_benhaddou",
      "nom": "Aït Benhaddou",
      "ville": "Ouarzazate (région)",
      "coordonnees": {"lat": 31.0472, "lon": -7.1290},
      "epoque": "XVIIe siècle",
      "style": "Architecture Ksourienne",
      "classification": "UNESCO (1987)",
      "histoire": "Aït Benhaddou est un ksar (village fortifié) exceptionnel situé le long de l'ancienne route caravanière entre le Sahara et Marrakech. Inscrit à l'UNESCO en 1987, il a servi de décor à de nombreuses productions cinématographiques mondiales.",
      "images": ["ait_benhaddou/ait_benhaddou_1.jpg"]
    },
    {
      "id": "medersa_bou_inania",
      "nom": "Medersa Bou Inania",
      "ville": "Fès",
      "coordonnees": {"lat": 34.0658, "lon": -4.9740},
      "epoque": "XIVe siècle (1351-1356)",
      "style": "Mérinide",
      "classification": "Monument Historique National",
      "histoire": "La Medersa Bou Inania est une école coranique mérinide construite entre 1351 et 1356 par le Sultan Abou Inan Faris. Chef-d'œuvre de l'architecture hispano-mauresque, elle est la seule medersa de Fès à posséder un minbar (chaire) et à fonctionner comme mosquée.",
      "images": ["medersa_bou_inania/bou_inania_1.jpg"]
    }
  ]
}
```

>  **Astuce :** Pour le TP, vous pouvez utiliser des images libres de droits téléchargées depuis Wikimedia Commons ou Unsplash.

---

---

## Cellule 0 — Création de l'environnement virtuel (venv)

> **A faire EN PREMIER, une seule fois, dans un terminal.**  
> Le venv isole les dépendances du projet et évite les conflits de versions.


In [1]:
# ============================================================
#  CELLULE 0 : Création et activation du venv
# ============================================================
# Exécutez les commandes ci-dessous dans un TERMINAL
# (pas dans Jupyter) AVANT de lancer ce notebook.
#
# Ces commandes sont affichées ici pour référence.
# La cellule suivante peut aussi créer le venv via Python
# si vous êtes sur un environnement qui le permet.
# ============================================================

import sys
import subprocess
from pathlib import Path

VENV_DIR = Path("venv_visual_rag")

# --- Affichage des commandes à exécuter dans le terminal ---
print("=" * 60)
print("  COMMANDES TERMINAL — Exécuter UNE SEULE FOIS")
print("=" * 60)
print()
print("  # 1. Création du venv")
print("  python -m venv venv_visual_rag")
print()
print("  # 2. Activation (Linux / macOS)")
print("  source venv_visual_rag/bin/activate")
print()
print("  # 2. Activation (Windows)")
print("  venv_visual_rag\\Scripts\\activate")
print()
print("  # 3. Relancer Jupyter dans le venv")
print("  pip install jupyter")
print("  jupyter notebook")
print("=" * 60)
print()

# --- Création automatique si pas encore existant ---
if not VENV_DIR.exists():
    print(f"Création du venv : {VENV_DIR} ...")
    subprocess.check_call([sys.executable, "-m", "venv", str(VENV_DIR)])
    print("venv créé avec succès.")
    print()
    print("Activez-le dans votre terminal, puis relancez Jupyter.")
else:
    print(f"venv '{VENV_DIR}' déjà existant.")

# --- Vérification que l'on tourne bien dans le venv ---
in_venv = (
    hasattr(sys, 'real_prefix') or
    (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
)
print()
if in_venv:
    print("Environnement virtuel actif :", sys.prefix)
else:
    print("ATTENTION : Vous n'êtes pas dans un venv.")
    print("Activez venv_visual_rag avant de continuer (voir commandes ci-dessus).")


  COMMANDES TERMINAL — Exécuter UNE SEULE FOIS

  # 1. Création du venv
  python -m venv venv_visual_rag

  # 2. Activation (Linux / macOS)
  source venv_visual_rag/bin/activate

  # 2. Activation (Windows)
  venv_visual_rag\Scripts\activate

  # 3. Relancer Jupyter dans le venv
  pip install jupyter
  jupyter notebook

venv 'venv_visual_rag' déjà existant.

Environnement virtuel actif : c:\Users\farou\Documents\etudes\S6\NLP\Hackaton\venv_visual_rag


##  Cellule 1 — Installation des dépendances et importations

In [2]:
# ============================================================
#  CELLULE 1 : Installation des dépendances
# ============================================================
# Exécutez cette cellule une seule fois. Si vous travaillez
# dans un environnement virtuel, assurez-vous qu'il est activé.
# Temps d'installation estimé : ~3-5 minutes (premier lancement)
# ============================================================

# sentence-transformers : contient les modèles CLIP pré-entraînés
# chromadb              : base de données vectorielle légère et locale
# langchain + ollama    : interface de haut niveau pour le LLM local
# Pillow                : traitement des images
# tqdm                  : barres de progression pédagogiques

import subprocess
import sys

packages = [
    "sentence-transformers",
    "chromadb",
    "langchain",
    "langchain-ollama",
    "langchain-community",
    "Pillow",
    "tqdm",
    "requests",
    "numpy",
]

print("Installation des paquets nécessaires...\n")
# Installation en une seule commande pip (plus rapide que boucle)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
)
print("\nToutes les dépendances sont installées !")


Installation des paquets nécessaires...


Toutes les dépendances sont installées !


In [36]:
# ============================================================
#  CELLULE 1b : Importations et configuration initiale
# ============================================================

# --- Bibliothèques standard ---
import os
import json
import warnings
from pathlib import Path

# --- Traitement d'image ---
from PIL import Image
import numpy as np

# --- Modèle d'embedding CLIP ---
from sentence_transformers import SentenceTransformer

# --- Base de données vectorielle ---
import chromadb

# --- Interface LLM ---
import requests
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# --- Utilitaires ---
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ============================================================
#  DÉTECTION AUTOMATIQUE DU MODÈLE OLLAMA DISPONIBLE
# ============================================================

def detecter_modele_ollama(ollama_url: str = "http://localhost:11434") -> str:
    """Retourne le premier modèle installé, ou lève une erreur claire."""
    try:
        r = requests.get(f"{ollama_url}/api/tags", timeout=5)
        r.raise_for_status()
        modeles = r.json().get("models", [])
        if not modeles:
            raise RuntimeError("Ollama tourne mais aucun modèle n'est installé.\n"
                               "  → Lancez : ollama pull phi3")
        noms = [m["name"] for m in modeles]
        print(f"Modèles Ollama détectés : {noms}")
        return noms[0]   # Prend le premier disponible
    except requests.exceptions.ConnectionError:
        raise RuntimeError("Ollama n'est pas lancé.\n"
                           "  → Lancez : ollama serve")

# ============================================================
#  CONFIGURATION GLOBALE
# ============================================================

OLLAMA_URL = "http://localhost:11434"

CONFIG = {
    "DATASET_PATH"   : Path("dataset"),
    "METADATA_PATH"  : Path("metadata.json"),
    "CHROMA_PATH"    : Path("chroma_db"),
    "COLLECTION_NAME": "lieux_touristiques_maroc",
    "CLIP_MODEL"     : "clip-ViT-B-32",
    "OLLAMA_URL"     : OLLAMA_URL,
    "LLM_MODEL"      : detecter_modele_ollama(OLLAMA_URL),  # ← détection auto
    "N_RESULTS"      : 3,
}

print("\nConfiguration chargée avec succès !")
print("\nParamètres actifs :")
for key, value in CONFIG.items():
    print(f"   {key:20s} = {value}")

Modèles Ollama détectés : ['llama3.2:latest', 'qwen2.5:1.5b']

Configuration chargée avec succès !

Paramètres actifs :
   DATASET_PATH         = dataset
   METADATA_PATH        = metadata.json
   CHROMA_PATH          = chroma_db
   COLLECTION_NAME      = lieux_touristiques_maroc
   CLIP_MODEL           = clip-ViT-B-32
   OLLAMA_URL           = http://localhost:11434
   LLM_MODEL            = llama3.2:latest
   N_RESULTS            = 3


---

##  Cellule 2 — Vérification de la connexion Ollama

> **Prérequis :** Ollama doit être installé et en cours d'exécution sur votre machine.  
> Installation : https://ollama.ai  
> Lancer Ollama : `ollama serve` (dans un terminal séparé)  
> Télécharger le modèle : `ollama pull llama3`

In [5]:
# ============================================================
#  CELLULE 2 : Vérification de la connexion Ollama
# ============================================================
# Cette cellule vérifie que :
#   1. Le serveur Ollama est bien démarré
#   2. Le modèle LLM souhaité est disponible localement
# ============================================================

def check_ollama_connection(config: dict) -> bool:
    """
    Vérifie la disponibilité du serveur Ollama et du modèle LLM.

    Args:
        config (dict): Dictionnaire de configuration global.

    Returns:
        bool: True si tout est opérationnel, False sinon.
    """
    base_url = config["OLLAMA_URL"]
    model_name = config["LLM_MODEL"]

    # --- Étape 1 : Vérification que le serveur répond ---
    print(f" Test de connexion à Ollama : {base_url}")
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=5)
        response.raise_for_status()
        print("    Serveur Ollama accessible !")
    except requests.exceptions.ConnectionError:
        print("    ERREUR : Impossible de joindre Ollama.")
        print("   → Assurez-vous que 'ollama serve' est lancé dans un terminal.")
        return False
    except requests.exceptions.Timeout:
        print("    ERREUR : Timeout — Ollama met trop de temps à répondre.")
        return False

    # --- Étape 2 : Vérification que le modèle est téléchargé ---
    print(f"\n Vérification du modèle '{model_name}'...")
    data = response.json()
    modeles_disponibles = [m["name"] for m in data.get("models", [])]

    if not modeles_disponibles:
        print("     Aucun modèle trouvé. Téléchargez-en un avec :")
        print(f"      ollama pull {model_name}")
        return False

    print(f"    Modèles disponibles : {', '.join(modeles_disponibles)}")

    # On vérifie si le modèle exact (ou une version) est présent
    model_found = any(model_name.split(":")[0] in m for m in modeles_disponibles)

    if model_found:
        print(f"    Modèle '{model_name}' prêt à l'emploi !")
        return True
    else:
        print(f"     Modèle '{model_name}' non trouvé. Commande pour le télécharger :")
        print(f"      ollama pull {model_name}")
        print("   ℹ  Alternativement, modifiez CONFIG['LLM_MODEL'] avec un modèle disponible.")
        return False


# --- Exécution de la vérification ---
ollama_ok = check_ollama_connection(CONFIG)

if ollama_ok:
    print("\n Configuration Ollama validée. Vous pouvez continuer !")
else:
    print("\n  Réglez le problème Ollama avant de continuer les cellules de génération.")
    print("    Les cellules d'indexation et de recherche fonctionneront quand même.")

 Test de connexion à Ollama : http://localhost:11434
    Serveur Ollama accessible !

 Vérification du modèle 'llama3'...
    Modèles disponibles : llama3.2:latest, qwen2.5:1.5b
    Modèle 'llama3' prêt à l'emploi !

 Configuration Ollama validée. Vous pouvez continuer !


---

##  Cellule 3 — Chargement du modèle CLIP

> **Note :** Le téléchargement de CLIP se fait automatiquement lors du premier lancement (~400 Mo).  
> Les exécutions suivantes utilisent le cache local (instantané).

In [7]:
# ============================================================
#  CELLULE 3 : Chargement du modèle CLIP
# ============================================================
# SentenceTransformer fournit un wrapper pratique pour CLIP.
# Le modèle 'clip-ViT-B-32' génère des vecteurs de dimension 512
# pour des images ET du texte dans le même espace latent.
# ============================================================

print(f"Chargement du modèle CLIP : '{CONFIG['CLIP_MODEL']}'")
print("   (Premier lancement : téléchargement ~400 Mo — patience !)\n")

# SentenceTransformer gère automatiquement le GPU si disponible (CUDA/MPS)
clip_model = SentenceTransformer(CONFIG["CLIP_MODEL"])

print("Modèle CLIP chargé avec succès !")

# --- Vérification des dimensions ---
img_test = Image.new('RGB', (224, 224), color=(128, 128, 128))  # Image grise 224x224
test_vector = clip_model.encode(img_test)

print(f"\nInformations sur les embeddings CLIP :")
print(f"   -> Dimension du vecteur : {test_vector.shape[0]}D")
print(f"   -> Type de données      : {test_vector.dtype}")
print(f"   -> Norme L2 (normalisée): {np.linalg.norm(test_vector):.4f}")

# --- Démonstration de l'espace latent multimodal ---
print("\nDémonstration de l'alignement image-texte :")

# Encodage en batch (une seule passe CLIP — plus efficace que deux appels séparés)
texte_test = "minaret d'une mosquée islamique au Maroc"
vec_texte = clip_model.encode(texte_test)
vec_image = clip_model.encode(np.array(img_test))

# Calcul de la similarité cosinus entre l'image grise (neutre) et le texte
similarite = np.dot(vec_texte, vec_image) / (
    np.linalg.norm(vec_texte) * np.linalg.norm(vec_image)
)
print(f"   Similarité cosinus (image neutre <-> texte 'minaret') : {similarite:.4f}")
print("   -> Faible car l'image grise n'a pas de contenu visuel significatif.")
print("   -> Avec une vraie photo de Tour Hassan, la similarité sera bien plus haute !")


Chargement du modèle CLIP : 'clip-ViT-B-32'
   (Premier lancement : téléchargement ~400 Mo — patience !)



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Modèle CLIP chargé avec succès !

Informations sur les embeddings CLIP :
   -> Dimension du vecteur : 512D
   -> Type de données      : float32
   -> Norme L2 (normalisée): 11.8174

Démonstration de l'alignement image-texte :
   Similarité cosinus (image neutre <-> texte 'minaret') : 0.2002
   -> Faible car l'image grise n'a pas de contenu visuel significatif.
   -> Avec une vraie photo de Tour Hassan, la similarité sera bien plus haute !


---

##  Cellule 3b — Initialisation de ChromaDB

> **ChromaDB** est une base de données vectorielle *embarquée* (pas de serveur séparé nécessaire).  
> Elle stocke vos vecteurs sur disque et permet des recherches par similarité cosinus très rapides.

In [14]:
# ============================================================
#  CELLULE 3b : Initialisation de ChromaDB
# ============================================================
# ChromaDB fonctionne en mode 'persistant' : les vecteurs sont
# sauvegardés sur disque dans le dossier CONFIG["CHROMA_PATH"].
# Vous n'aurez donc à indexer les images qu'une seule fois.
# ============================================================

# Création du dossier de persistance si nécessaire
CONFIG["CHROMA_PATH"].mkdir(parents=True, exist_ok=True)

# Initialisation du client ChromaDB persistant
chroma_client = chromadb.PersistentClient(
    path=str(CONFIG["CHROMA_PATH"])
)

# Création (ou récupération si elle existe déjà) de la collection
# IMPORTANT : get_or_create_collection évite les doublons entre exécutions
collection = chroma_client.get_or_create_collection(
    name=CONFIG["COLLECTION_NAME"],
    # ChromaDB gère nativement la similarité cosinus
    metadata={"hnsw:space": "cosine"}
)

print(" ChromaDB initialisé avec succès !")
print(f"   → Dossier de persistance : {CONFIG['CHROMA_PATH'].resolve()}")
print(f"   → Collection             : '{CONFIG['COLLECTION_NAME']}'")
print(f"   → Documents indexés      : {collection.count()}")

if collection.count() > 0:
    print("\n  Attention : la collection contient déjà des documents.")
    print("   Si vous souhaitez réindexer from scratch, exécutez :")
    print(f"   chroma_client.delete_collection('{CONFIG['COLLECTION_NAME']}')")

 ChromaDB initialisé avec succès !
   → Dossier de persistance : C:\Users\farou\Documents\etudes\S6\NLP\Hackaton\chroma_db
   → Collection             : 'lieux_touristiques_maroc'
   → Documents indexés      : 0


---

##  Cellule 4 — EXERCICE : Fonction d'indexation des images

```
┌─────────────────────────────────────────────────────────────────────┐
│  OBJECTIF DE CET EXERCICE                                           │
│                                                                     │
│  Implémenter la Phase d'Ingestion du pipeline Visual RAG :          │
│                                                                     │
│  Pour chaque lieu dans metadata.json :                              │
│    1. Charger chaque image associée avec Pillow                     │
│    2. Générer son embedding CLIP (vecteur 512D)                     │
│    3. Stocker le vecteur + métadonnées dans ChromaDB                │
│                                                                     │
│  Fonctions à compléter : index_images()                             │
└─────────────────────────────────────────────────────────────────────┘
```

**Concepts à maîtriser :**
- `Image.open()` → Chargement d'une image depuis un chemin
- `clip_model.encode(image)` → Génération d'un embedding (retourne un `np.array`)
- `collection.add(ids, embeddings, metadatas, documents)` → Insertion dans ChromaDB

**Documentation ChromaDB :**
```python
collection.add(
    ids=["id_unique_1", "id_unique_2"],           # Identifiants uniques (strings)
    embeddings=[[0.1, 0.2, ...], [0.3, 0.4, ...]],# Vecteurs (listes de floats)
    metadatas=[{"clé": "valeur"}, {...}],          # Métadonnées (dicts)
    documents=["texte1", "texte2"]                 # Textes lisibles (pour debug)
)
```

In [17]:
# ============================================================
#  CELLULE 4 : EXERCICE — Fonction d'indexation
# ============================================================
#
#  CONSIGNES :
#  Complétez les parties marquées "# TODO" ci-dessous.
#  Ne modifiez pas la signature de la fonction.
#  Les commentaires sont là pour vous guider, pas pour donner
#  la solution directement.
#
# ============================================================

def charger_metadata(metadata_path: Path) -> dict:
    """
    Charge et retourne le contenu du fichier metadata.json.
    Cette fonction est fournie — pas de modification nécessaire.
    """
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)


def index_images(
    clip_model: SentenceTransformer,
    collection,
    metadata: dict,
    dataset_path: Path,
    batch_size: int = 32,
) -> int:
    """
    Indexe toutes les images de référence dans ChromaDB.

    Pour chaque lieu dans les métadonnées :
      - Charge chaque image depuis le disque.
      - Génère un embedding CLIP.
      - Stocke le vecteur + les informations du lieu dans ChromaDB
        par lots (batch) pour limiter les écritures disque.

    Args:
        clip_model        : Le modèle CLIP chargé.
        collection        : La collection ChromaDB.
        metadata (dict)   : Les métadonnées chargées depuis metadata.json.
        dataset_path (Path): Le chemin racine vers le dossier 'dataset/'.
        batch_size (int)  : Nombre d'images insérées par lot dans ChromaDB.

    Returns:
        int : Le nombre total d'images indexées.
    """
    total_indexed = 0
    lieux = metadata["lieux"]

    print(f"Début de l'indexation ({len(lieux)} lieux, Phase Ingestion)...\n")

    # Accumulateurs pour l'insertion par lots
    batch_ids, batch_embeddings, batch_metadatas, batch_documents = [], [], [], []

    def flush_batch():
        """Vide le lot courant dans ChromaDB."""
        if batch_ids:
            collection.add(
                ids=batch_ids,
                embeddings=batch_embeddings,
                metadatas=batch_metadatas,
                documents=batch_documents,
            )
            batch_ids.clear()
            batch_embeddings.clear()
            batch_metadatas.clear()
            batch_documents.clear()

    # --- Boucle principale sur chaque lieu ---
    for lieu in tqdm(lieux, desc="Lieux traités"):

        # Récupération des informations du lieu depuis le JSON
        lieu_id    = lieu["id"]
        lieu_nom   = lieu["nom"]
        lieu_hist  = lieu["histoire"]
        lieu_ville = lieu["ville"]
        lieu_imgs  = lieu["images"]  # Liste de chemins relatifs

        # --- Boucle sur chaque image du lieu ---
        for i, img_relative_path in enumerate(lieu_imgs):

            # Construction du chemin complet vers l'image
            img_path = dataset_path / img_relative_path

            # Vérification que le fichier existe avant de continuer
            if not img_path.exists():
                print(f"  Image introuvable : {img_path} — ignorée.")
                continue

            # --------------------------------------------------
            # TODO 1 : Chargez l'image avec Pillow
            # Indice : utilisez Image.open() et .convert('RGB')
            # La conversion en RGB est importante pour les images
            # avec canal alpha (PNG) ou en niveaux de gris.
            # --------------------------------------------------
            image = Image.open(img_path).convert("RGB")


            # --------------------------------------------------
            # TODO 2 : Générez l'embedding CLIP de l'image
            # Indice : utilisez clip_model.encode()
            # Le résultat est un np.array de dimension 512.
            # ChromaDB attend une liste Python, pas un np.array.
            # Pensez à convertir avec .tolist()
            # --------------------------------------------------
            embedding = clip_model.encode(np.array(image)).tolist()


            # --------------------------------------------------
            # TODO 3 : Créez un identifiant unique pour cette image
            # Convention suggérée : "{lieu_id}_img_{i}"
            # Exemple : "tour_hassan_img_0", "tour_hassan_img_1"
            # L'identifiant doit être une chaîne de caractères.
            # --------------------------------------------------
            doc_id = f"{lieu_id}_img_{i}"


            # --------------------------------------------------
            # TODO 4 : Construisez le dictionnaire de métadonnées
            # C'est ce contexte que nous récupérerons lors de la
            # recherche pour l'envoyer au LLM.
            # Incluez au minimum : nom, ville, histoire, lieu_id
            # ATTENTION : ChromaDB n'accepte que str, int, float, bool
            # dans les métadonnées. Pas de listes ni de dicts imbriqués.
            # --------------------------------------------------
            metadonnees = {
                "nom": lieu_nom,
                "ville": lieu_ville,
                "histoire": lieu_hist,
                "lieu_id": lieu_id,
            }


            # --------------------------------------------------
            # TODO 5 : Ajoutez le vecteur dans le lot (batch)
            # Ne pas appeler collection.add() directement ici —
            # l'insertion par lots se fait via flush_batch().
            # --------------------------------------------------
            batch_ids.append(doc_id)
            batch_embeddings.append(embedding)
            batch_metadatas.append(metadonnees)
            batch_documents.append(lieu_nom)


            total_indexed += 1

            # Insertion automatique quand le lot est plein
            if len(batch_ids) >= batch_size:
                flush_batch()

    # Insérer le reste (lot partiel final)
    flush_batch()

    print(f"\nIndexation terminée !")
    print(f"   -> {total_indexed} images indexées dans ChromaDB.")
    print(f"   -> Collection '{collection.name}' : {collection.count()} documents au total.")
    
    return total_indexed


# ============================================================
#  TEST de la fonction (décommentez quand votre code est prêt)
# ============================================================

metadata = charger_metadata(CONFIG["METADATA_PATH"])
n_indexed = index_images(clip_model, collection, metadata, CONFIG["DATASET_PATH"])

print("Fonction d'indexation définie. Complétez les TODO et décommentez le test !")

Début de l'indexation (5 lieux, Phase Ingestion)...



Lieux traités: 100%|██████████| 5/5 [00:00<00:00,  5.83it/s]


Indexation terminée !
   -> 8 images indexées dans ChromaDB.
   -> Collection 'lieux_touristiques_maroc' : 8 documents au total.
Fonction d'indexation définie. Complétez les TODO et décommentez le test !


###  Validation de l'indexation

Une fois l'indexation terminée, utilisez cette cellule pour vérifier que vos données sont bien stockées dans ChromaDB.

In [18]:
# ============================================================
#  CELLULE 4c : Validation de l'indexation
# ============================================================
# Exécutez cette cellule pour inspecter ce qui est stocké.
# ============================================================

def inspecter_collection(collection) -> None:
    """Affiche un aperçu du contenu de la collection ChromaDB."""
    count = collection.count()
    print(f" Inspection de la collection '{collection.name}'")
    print(f"   Total de documents indexés : {count}")

    if count == 0:
        print("     La collection est vide. Exécutez d'abord la cellule d'indexation.")
        return

    # Récupération d'un échantillon pour inspection
    echantillon = collection.get(
        limit=5,
        include=["metadatas", "documents"]
    )

    print("\n Échantillon (5 premiers documents) :")
    print("-" * 60)
    for doc_id, doc, meta in zip(
        echantillon["ids"],
        echantillon["documents"],
        echantillon["metadatas"]
    ):
        print(f"  ID       : {doc_id}")
        print(f"  Lieu     : {doc}")
        print(f"  Ville    : {meta.get('ville', 'N/A')}")
        print(f"  Histoire : {meta.get('histoire', 'N/A')[:80]}...")
        print("-" * 60)

inspecter_collection(collection)

 Inspection de la collection 'lieux_touristiques_maroc'
   Total de documents indexés : 8

 Échantillon (5 premiers documents) :
------------------------------------------------------------
  ID       : tour_hassan_img_0
  Lieu     : Tour Hassan
  Ville    : Rabat
  Histoire : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque comman...
------------------------------------------------------------
  ID       : tour_hassan_img_1
  Lieu     : Tour Hassan
  Ville    : Rabat
  Histoire : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque comman...
------------------------------------------------------------
  ID       : volubilis_img_0
  Lieu     : Volubilis
  Ville    : Meknès (région)
  Histoire : Volubilis est une ancienne cité berbère et romaine partiellement fouillée. Capit...
------------------------------------------------------------
  ID       : volubilis_img_1
  Lieu     : Volubilis
  Ville    : Meknès (région)
  Histoire : Volubilis e

---

##  Cellule 5 — EXERCICE : Fonction de recherche (Retrieval)

```
┌─────────────────────────────────────────────────────────────────────┐
│  OBJECTIF DE CET EXERCICE                                           │
│                                                                     │
│  Implémenter la Phase de Retrieval du pipeline Visual RAG :         │
│                                                                     │
│  Étant donné une nouvelle image (non indexée) :                     │
│    1. Générer son embedding CLIP (même espace vectoriel)            │
│    2. Interroger ChromaDB avec cet embedding                        │
│    3. Retourner les N lieux les plus similaires avec leurs          │
│       métadonnées (nom, histoire, score de similarité)              │
│                                                                     │
│  Fonction à compléter : rechercher_lieu()                           │
└─────────────────────────────────────────────────────────────────────┘
```

**Concept clé — Similarité Cosinus :**

```
  vecteur_A · vecteur_B
  ───────────────────────  =  cos(θ)  ∈ [-1, 1]
  ‖vecteur_A‖ × ‖vecteur_B‖

  cos(θ) = 1.0  → Vecteurs identiques   (même lieu, même angle)
  cos(θ) = 0.0  → Vecteurs orthogonaux  (lieux sans rapport)
  cos(θ) < 0.0  → Opposés (rare avec des embeddings normalisés)
```

**Documentation ChromaDB — Query :**
```python
results = collection.query(
    query_embeddings=[[0.1, 0.2, ...]],  # Vecteur de la requête
    n_results=3,                          # Nombre de voisins à retourner
    include=["metadatas", "documents", "distances"]  # Que récupérer
)
# results est un dict avec les clés 'ids', 'distances', 'metadatas', 'documents'
# Chaque valeur est une liste de listes (une sous-liste par query)
```

In [21]:
# ============================================================
#  CELLULE 5 : EXERCICE — Fonction de recherche (Retrieval)
# ============================================================
#
#    CONSIGNES :
#  Complétez les parties marquées "# TODO" ci-dessous.
#  Votre fonction doit retourner une liste de dictionnaires
#  selon le format spécifié dans la docstring.
#
# ============================================================

def rechercher_lieu(
    image_path: str | Path,
    clip_model: SentenceTransformer,
    collection,
    n_results: int = 3
) -> list[dict]:
    """
    Recherche les lieux touristiques les plus similaires à une image requête.
    """

    print(f" Recherche en cours pour : {Path(image_path).name}")

    # --------------------------------------------------
    # TODO 1 : Chargez l'image de requête avec Pillow
    # Indice : même approche que dans index_images()
    # Convertissez en RGB pour garantir la compatibilité
    # --------------------------------------------------
    image_requete = Image.open(image_path).convert("RGB")


    # --------------------------------------------------
    # TODO 2 : Générez l'embedding CLIP de l'image requête
    # ATTENTION : Le vecteur doit être une liste Python
    # (pas un np.array) pour ChromaDB.
    # Indice : .tolist() convertit un np.array en liste
    # --------------------------------------------------
    query_embedding = clip_model.encode(np.array(image_requete)).tolist()


    # --------------------------------------------------
    # TODO 3 : Interrogez ChromaDB avec l'embedding
    # Utilisez collection.query()
    # N'oubliez pas d'inclure metadatas, documents ET distances
    # dans le paramètre 'include' pour récupérer toutes les infos
    # --------------------------------------------------
    resultats_bruts = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["metadatas", "documents", "distances"]
    )


    # --------------------------------------------------
    # TODO 4 : Formatez les résultats
    # ChromaDB retourne des listes de listes (une sous-liste
    # par requête). Comme nous avons 1 seule requête :
    #   resultats_bruts["metadatas"][0]  → liste des métadonnées
    #   resultats_bruts["distances"][0]  → liste des distances
    #   resultats_bruts["documents"][0]  → liste des noms de lieux
    #
    # La distance ChromaDB (cosinus) est dans [0, 2].
    # Convertissez en score de similarité : score = 1 - distance
    # (score ∈ [-1, 1], où 1 = identique, 0 = orthogonal)
    # --------------------------------------------------
    resultats_formates = []

    for rang, (meta, dist, doc) in enumerate(
        zip(
            resultats_bruts["metadatas"][0],
            resultats_bruts["distances"][0],
            resultats_bruts["documents"][0]
        ),
        start=1
    ):
        score = 1 - dist  # Conversion distance → similarité

        resultats_formates.append({
            "rang"    : rang,
            "nom"     : meta.get("nom", "Inconnu"),
            "ville"   : meta.get("ville", "N/A"),
            "histoire": meta.get("histoire", ""),
            "score"   : round(score, 4),
            "distance": round(dist, 4),
        })

    return resultats_formates


# ============================================================
#  Fonction d'affichage des résultats (fournie)
# ============================================================

def afficher_resultats(resultats: list[dict]) -> None:
    """Affiche les résultats de recherche de façon lisible."""
    if not resultats:
        print("  Aucun résultat retourné. Vérifiez votre indexation.")
        return

    print(f"\n{'═' * 65}")
    print(f"   RÉSULTATS DE LA RECHERCHE VISUELLE")
    print(f"{'═' * 65}")

    for r in resultats:
        emoji = "" if r["rang"] == 1 else ("" if r["rang"] == 2 else "")
        barre = "█" * int(r["score"] * 30) + "░" * (30 - int(r["score"] * 30))

        print(f"\n  {emoji} Rang {r['rang']} : {r['nom']} ({r['ville']})")
        print(f"     Score  : [{barre}] {r['score']:.4f}")
        print(f"     Aperçu : {r['histoire'][:100]}...")

    print(f"\n{'═' * 65}")


# ============================================================
#  TEST de la fonction (décommentez quand votre code est prêt)
# ============================================================

IMAGE_DE_TEST = "test_images/ma_photo_tour_hassan.png"
resultats = rechercher_lieu(IMAGE_DE_TEST, clip_model, collection, n_results=CONFIG["N_RESULTS"])
afficher_resultats(resultats)

print(" Fonction de recherche définie. Complétez les TODO et décommentez le test !")

 Recherche en cours pour : ma_photo_tour_hassan.png

═════════════════════════════════════════════════════════════════
   RÉSULTATS DE LA RECHERCHE VISUELLE
═════════════════════════════════════════════════════════════════

   Rang 1 : Tour Hassan (Rabat)
     Score  : [█████████████████████████░░░░░] 0.8605
     Aperçu : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque commandée par le Sultan Ya...

   Rang 2 : Tour Hassan (Rabat)
     Score  : [█████████████████████░░░░░░░░░] 0.7261
     Aperçu : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque commandée par le Sultan Ya...

   Rang 3 : Medersa Bou Inania (Fès)
     Score  : [████████████████████░░░░░░░░░░] 0.6983
     Aperçu : La Medersa Bou Inania est une école coranique mérinide construite entre 1351 et 1356 par le Sultan A...

═════════════════════════════════════════════════════════════════
 Fonction de recherche définie. Complétez les TODO et décommentez le test !


---

##  Cellule 6 — Intégration avec Ollama (Génération)

> **Cette cellule est fournie.** Elle illustre la Phase d'Augmentation et de Génération.  
> Deux approches sont proposées : via `requests` (bas niveau) et via LangChain (haut niveau).

```
  PHASE D'AUGMENTATION — Construction du prompt enrichi
  ────────────────────────────────────────────────────────

  Contexte (issu du Retrieval)      Question de l'utilisateur
         │                                    │
         ▼                                    ▼
  ┌─────────────────────────────────────────────────────┐
  │  [SYSTEM PROMPT]                                    │
  │  Tu es un guide touristique expert au Maroc...      │
  │                                                     │
  │  [CONTEXTE RÉCUPÉRÉ]                                │
  │  Lieu identifié : Tour Hassan                       │
  │  Ville : Rabat                                      │
  │  Histoire : La Tour Hassan est le minaret...        │
  │                                                     │
  │  [QUESTION]                                         │
  │  Parle-moi de ce lieu touristique.                  │
  └───────────────────────┬─────────────────────────────┘
                          │
                          ▼
                   Ollama (LLM local)
                          │
                          ▼
               Réponse en langage naturel
               fluide et informative 
```

In [37]:
# ============================================================
#  CELLULE 6 : Génération avec Ollama (via requests)
# ============================================================
# Cette approche utilise directement l'API REST d'Ollama.
# Avantage : pas de dépendance supplémentaire, contrôle total.
# ============================================================

SYSTEM_PROMPT = """
Tu es un guide touristique expert et passionné spécialisé dans le patrimoine 
historique et culturel du Maroc. Tu t'exprimes de manière élégante, pédagogique
et engageante, comme si tu guidais un visiteur sur place.

RÈGLES IMPORTANTES :
- Utilise UNIQUEMENT les informations fournies dans le contexte.
- Ne réponds pas aux questions qui n'ont pas de rapport avec le tourisme ou l'histoire.
- Si le contexte est insuffisant, dis-le honnêtement.
- Structure ta réponse : présentation -> histoire -> anecdote -> conseil visite.
- Réponds TOUJOURS en français.
""".strip()


def construire_prompt_augmente(contexte: dict, question_utilisateur: str) -> str:
    """
    Construit le prompt enrichi (Phase Augmentation).
    Combine les métadonnées récupérées et la question de l'utilisateur.

    Args:
        contexte (dict)            : Métadonnées du lieu identifié (1er résultat).
        question_utilisateur (str) : Question posée par l'utilisateur.

    Returns:
        str : Prompt complet prêt à envoyer au LLM.
    """
    prompt = f"""
=== INFORMATIONS SUR LE LIEU IDENTIFIÉ ===
Nom officiel  : {contexte.get('nom', 'Non renseigné')}
Ville / Région: {contexte.get('ville', 'Non renseignée')}
Score de confiance: {contexte.get('score', 0):.2%}
Contexte historique:
{contexte.get('histoire', 'Aucune information disponible.')}
==========================================

QUESTION DU VISITEUR : {question_utilisateur}
""".strip()
    return prompt


def generate_response(
    context: dict,
    user_query: str,
    ollama_url: str = CONFIG["OLLAMA_URL"],
    model: str = CONFIG["LLM_MODEL"],
    timeout: int = 120,
) -> str:
    """
    Génère une réponse en langage naturel via Ollama (Phase Génération).

    Args:
        context (dict)   : Dictionnaire de métadonnées du lieu identifié.
        user_query (str) : Question posée par l'utilisateur.
        ollama_url (str) : URL du serveur Ollama.
        model (str)      : Nom du modèle Ollama à utiliser.
        timeout (int)    : Délai max en secondes avant abandon (défaut : 120).

    Returns:
        str : Réponse générée par le LLM.
    """
    # --- Construction du prompt augmenté ---
    prompt_complet = construire_prompt_augmente(context, user_query)

    # --- Préparation de la requête API Ollama ---
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_complet}
        ],
        "stream": False,           # False = attend la réponse complète
        "options": {
            "temperature": 0.7,    # Créativité modérée
            "top_p": 0.9,          # Nucleus sampling
            "num_predict": 512,    # Limite de tokens générés
        }
    }

    print(f"Envoi au LLM '{model}' via Ollama...")

    try:
        # Appel à l'endpoint /api/chat d'Ollama
        response = requests.post(
            f"{ollama_url}/api/chat",
            json=payload,
            timeout=timeout,
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.Timeout:
        return "ERREUR : Timeout — Le LLM met trop de temps. Essayez un modèle plus léger (phi3, gemma2:2b)."
    except requests.exceptions.ConnectionError:
        return "ERREUR : Connexion refusée. Vérifiez qu'Ollama est lancé ('ollama serve')."
    except (KeyError, ValueError) as e:
        return f"ERREUR : Format de réponse inattendu — {e}"


print("Fonctions de génération définies !")
print(f"   -> Modèle LLM configuré : '{CONFIG['LLM_MODEL']}'")
print(f"   -> URL Ollama           : '{CONFIG['OLLAMA_URL']}'")


Fonctions de génération définies !
   -> Modèle LLM configuré : 'llama3.2:latest'
   -> URL Ollama           : 'http://localhost:11434'


---

##  Cellule 7 — Pipeline Complet End-to-End

Assemblage de toutes les étapes : **Image → Retrieval → Augmentation → Génération**

In [38]:
# ============================================================
#  CELLULE 7 : Pipeline Visual RAG — Assemblage complet
# ============================================================
# Cette cellule orchestre toutes les étapes du pipeline
# en une seule fonction appelable.
# ============================================================

def visual_rag_pipeline(
    image_path: str | Path,
    question: str = "Parle-moi de ce lieu touristique. Quelle est son histoire ?",
    verbose: bool = True
) -> dict:
    """
    Pipeline Visual RAG complet.

    Étapes :
        1. Chargement de l'image
        2. Embedding CLIP (Vectorisation)
        3. Recherche ChromaDB (Retrieval)
        4. Augmentation du prompt
        5. Génération Ollama

    Args:
        image_path (str|Path): Chemin vers l'image à identifier.
        question (str)       : Question à poser sur le lieu identifié.
        verbose (bool)       : Affiche les étapes intermédiaires.

    Returns:
        dict : {
            'lieu_identifie' : dict (résultat #1 du retrieval),
            'tous_resultats' : list (tous les résultats),
            'reponse_llm'    : str (réponse générée)
        }
    """
    print("\n" + "=" * 65)
    print("  VISUAL RAG — IDENTIFICATION DE LIEU TOURISTIQUE")
    print("=" * 65)

    # --- Étape 1 & 2 & 3 : Retrieval ---
    print("\n[ÉTAPE 1-3] Retrieval...")
    resultats = rechercher_lieu(
        image_path=image_path,
        clip_model=clip_model,
        collection=collection,
        n_results=CONFIG["N_RESULTS"]
    )

    if not resultats:
        return {"erreur": "Aucun résultat trouvé. Base de données vide ?"}

    if verbose:
        afficher_resultats(resultats)

    # Le lieu le mieux identifié est le premier résultat
    lieu_principal = resultats[0]

    # --- Étape 4 : Augmentation & Génération ---
    print(f"\n[ÉTAPE 4-5] Génération de la réponse...")
    print(f"   Lieu identifié : {lieu_principal['nom']} "
          f"(confiance : {lieu_principal['score']:.2%})")
    print(f"   Question       : {question}")

    reponse = generate_response(
        context=lieu_principal,
        user_query=question
    )

    # --- Affichage final ---
    print("\n" + "=" * 65)
    print("  RÉPONSE DU GUIDE TOURISTIQUE IA")
    print("=" * 65)
    print(reponse)
    print("=" * 65)

    return {
        "lieu_identifie": lieu_principal,
        "tous_resultats": resultats,
        "reponse_llm"   : reponse
    }


# ============================================================
#  DÉMONSTRATION — Modifiez le chemin et la question
# ============================================================

# Remplacez par le chemin vers votre image de test
MON_IMAGE = "test_images/ma_photo_tour_hassan.png"
MA_QUESTION = "Raconte-moi l'histoire de ce lieu et donne-moi 3 conseils pour ma visite."

# Décommentez la ligne ci-dessous pour lancer le pipeline complet !
resultat = visual_rag_pipeline(MON_IMAGE, MA_QUESTION)

print("Pipeline prêt ! Décommentez la dernière ligne pour tester.")



  VISUAL RAG — IDENTIFICATION DE LIEU TOURISTIQUE

[ÉTAPE 1-3] Retrieval...
 Recherche en cours pour : ma_photo_tour_hassan.png

═════════════════════════════════════════════════════════════════
   RÉSULTATS DE LA RECHERCHE VISUELLE
═════════════════════════════════════════════════════════════════

   Rang 1 : Tour Hassan (Rabat)
     Score  : [█████████████████████████░░░░░] 0.8605
     Aperçu : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque commandée par le Sultan Ya...

   Rang 2 : Tour Hassan (Rabat)
     Score  : [█████████████████████░░░░░░░░░] 0.7261
     Aperçu : La Tour Hassan est le minaret inachevé d'une mosquée almohade gigantesque commandée par le Sultan Ya...

   Rang 3 : Medersa Bou Inania (Fès)
     Score  : [████████████████████░░░░░░░░░░] 0.6983
     Aperçu : La Medersa Bou Inania est une école coranique mérinide construite entre 1351 et 1356 par le Sultan A...

═════════════════════════════════════════════════════════════════

[ÉTAPE 4-5] 

---

##  Cellule 8 — Évaluation qualitative du système

Cette cellule vous permet de mesurer les performances de votre système sur plusieurs images de test.

In [40]:
# ============================================================
#  CELLULE 8 : Évaluation et analyse des performances
# ============================================================
# Testez votre système sur un ensemble d'images et calculez
# le taux de bonne identification (Precision@1).
# ============================================================

def evaluer_systeme(images_test: list[dict], verbose: bool = True) -> dict:
    """
    Évalue le système Visual RAG sur un ensemble de test.

    Args:
        images_test (list[dict]): Liste de {"chemin": str, "lieu_attendu": str}
        verbose (bool)          : Affichage détaillé.

    Returns:
        dict : Métriques d'évaluation (précision, scores moyens, etc.)
    """
    correct = 0
    scores  = []
    erreurs = []

    print(f"\n Évaluation sur {len(images_test)} images de test...\n")
    print("-" * 65)

    for test in images_test:
        chemin_img    = test["chemin"]
        lieu_attendu  = test["lieu_attendu"]

        # Recherche du lieu
        resultats = rechercher_lieu(chemin_img, clip_model, collection, n_results=1)

        if not resultats:
            print(f"   {Path(chemin_img).name:30s} → Aucun résultat")
            erreurs.append(chemin_img)
            continue

        lieu_predit = resultats[0]["nom"]
        score       = resultats[0]["score"]
        scores.append(score)

        # Vérification (insensible à la casse et aux espaces)
        est_correct = lieu_attendu.lower().strip() in lieu_predit.lower().strip()

        if est_correct:
            correct += 1
            statut = ""
        else:
            statut = ""

        if verbose:
            print(f"  {statut} {Path(chemin_img).name:30s} "
                  f"| Attendu: {lieu_attendu:20s} "
                  f"| Prédit: {lieu_predit:20s} "
                  f"| Score: {score:.3f}")

    # --- Calcul des métriques ---
    total   = len(images_test) - len(erreurs)
    precision = correct / total if total > 0 else 0
    score_moyen = np.mean(scores) if scores else 0

    print("-" * 65)
    print(f"\n RÉSULTATS D'ÉVALUATION :")
    print(f"   Precision@1 (Top-1 Accuracy) : {precision:.2%} ({correct}/{total})")
    print(f"   Score de similarité moyen    : {score_moyen:.4f}")
    print(f"   Images non traitées (erreurs): {len(erreurs)}")

    return {
        "precision_at_1": precision,
        "score_moyen"   : score_moyen,
        "correct"       : correct,
        "total"         : total,
        "erreurs"       : erreurs
    }


# ============================================================
#  Définissez votre ensemble de test
# ============================================================
#
# Exemple de format attendu :

IMAGES_DE_TEST = [
    {"chemin": "test_images/test_tour_hassan.png",    "lieu_attendu": "Tour Hassan"},
    {"chemin": "test_images/test_volubilis.png",       "lieu_attendu": "Volubilis"},
    {"chemin": "test_images/test_sefrou.png",          "lieu_attendu": "Cascades de Sefrou"},
    {"chemin": "test_images/test_ait_benhaddou.png",   "lieu_attendu": "Aït Benhaddou"},
]

# Décommentez pour lancer l'évaluation :
metriques = evaluer_systeme(IMAGES_DE_TEST)

print(" Système d'évaluation prêt. Ajoutez vos images de test et décommentez !")


 Évaluation sur 4 images de test...

-----------------------------------------------------------------
 Recherche en cours pour : test_tour_hassan.png
   test_tour_hassan.png           | Attendu: Tour Hassan          | Prédit: Tour Hassan          | Score: 0.861
 Recherche en cours pour : test_volubilis.png
   test_volubilis.png             | Attendu: Volubilis            | Prédit: Volubilis            | Score: 0.791
 Recherche en cours pour : test_sefrou.png
   test_sefrou.png                | Attendu: Cascades de Sefrou   | Prédit: Aït Benhaddou        | Score: 0.798
 Recherche en cours pour : test_ait_benhaddou.png
   test_ait_benhaddou.png         | Attendu: Aït Benhaddou        | Prédit: Cascades de Sefrou   | Score: 0.771
-----------------------------------------------------------------

 RÉSULTATS D'ÉVALUATION :
   Precision@1 (Top-1 Accuracy) : 50.00% (2/4)
   Score de similarité moyen    : 0.8051
   Images non traitées (erreurs): 0
 Système d'évaluation prêt. Ajoutez vos imag

---

##  Questions de réflexion & Pistes d'amélioration

### Questions théoriques

1. **Espace latent** : Pourquoi peut-on comparer directement un vecteur d'image et un vecteur de texte avec CLIP ? Quelle propriété de l'entraînement le permet ?

2. **Similarité cosinus vs distance euclidienne** : Dans quel cas préférerait-on l'une à l'autre pour des embeddings normalisés ?

3. **Hallucination** : Sans le mécanisme RAG, que se passerait-il si on demandait directement au LLM d'identifier le lieu sur la photo ? Quels risques cela pose-t-il ?

4. **Limites de CLIP** : CLIP a été entraîné principalement sur des images occidentales. Comment cela pourrait-il affecter les performances sur des sites patrimoniaux marocains moins représentés ?

---

### Pistes d'amélioration (Pour aller plus loin)

```
Niveau 1 — Basique
  ├── Augmenter la base de données (plus de lieux, plus d'images par lieu)
  ├── Ajouter la recherche textuelle hybride (image + mots-clés)
  └── Intégrer une interface Gradio ou Streamlit

Niveau 2 — Intermédiaire
  ├── Remplacer CLIP par un modèle fine-tuné sur le patrimoine marocain
  ├── Implémenter le streaming de réponse Ollama (stream=True)
  ├── Ajouter la mémoire conversationnelle (LangChain ConversationChain)
  └── Expérimenter avec différents modèles CLIP (ViT-L/14, SigLIP)

Niveau 3 — Avancé
  ├── Fine-tuning CLIP contrastif sur votre dataset patrimonial
  ├── Intégration multimodale : LLaVA / MiniCPM-V (LLM visuel local)
  ├── Reranking des résultats avec un Cross-Encoder
  └── Déploiement avec FastAPI + conteneurisation Docker
```

---

### Comparaison des approches

| Approche | Avantages | Inconvénients |
|----------|-----------|---------------|
| **Visual RAG (notre TP)** | Contrôle total, données maîtrisées, local | Nécessite une base de données manuelle |
| **LLM visuel (LLaVA)** | Pas de base de données nécessaire | Hallucinations, moins précis sur des sites rares |
| **API GPT-4o Vision** | Très performant, zero-shot | Coût, dépendance cloud, confidentialité |
| **Classification CNN classique** | Rapide, précis sur classes connues | Pas de génération de texte, inflexible |

---

##  Ressources pour approfondir

-  **Paper CLIP** : *"Learning Transferable Visual Models From Natural Language Supervision"* (Radford et al., 2021)
-  **Paper RAG** : *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"* (Lewis et al., 2020)
-  **ChromaDB docs** : https://docs.trychroma.com
-  **Ollama** : https://ollama.ai
-  **sentence-transformers CLIP** : https://www.sbert.net/docs/pretrained_cross-encoders.html
-  **Images libres de droits (Maroc)** : https://commons.wikimedia.org

---

```
  ╔══════════════════════════════════════════════════════╗
  ║   Bonne chance ! 🇲🇦                                 ║
  ║                                                      ║
  ║   "L'IA ne remplace pas le guide humain,             ║
  ║    elle lui donne une mémoire sans limite."          ║
  ╚══════════════════════════════════════════════════════╝
```

1. Pourquoi peut-on comparer image et texte avec CLIP ?
CLIP a été entraîné en regardant des millions de paires (photo + légende). Il a appris que photo d'un chat et texte "un chat" doivent donner des vecteurs proches. C'est comme si image et texte parlaient désormais la même langue mathématique — on peut donc les comparer directement.

2. Cosinus vs Euclidienne ?
Imagine deux flèches dans l'espace :

Cosinus regarde l'angle entre elles → "pointent-elles dans la même direction ?"
Euclidienne regarde la distance entre leurs pointes

Avec CLIP, toutes les flèches ont la même longueur (vecteurs normalisés), donc les deux donnent le même résultat. Mais on utilise quand même le cosinus par convention car il mesure mieux la similarité sémantique.

3. Sans RAG, que ferait le LLM ?
Il inventerait. Le LLM n'a jamais vu votre photo — il va juste produire quelque chose qui semble logique mais qui peut être totalement faux. Exemple : vous lui montrez la Tour Hassan, il vous parle de la Koutoubia avec des dates inventées, mais avec un ton très confiant. C'est ce qu'on appelle une hallucination. Le RAG évite ça en lui donnant les vraies infos avant qu'il réponde.

4. Limites de CLIP sur le patrimoine marocain ?
CLIP a surtout appris sur des photos et textes occidentaux (internet anglophone). Les monuments marocains y sont peu représentés, donc CLIP les "connaît" moins bien. Résultat : deux mosquées différentes peuvent avoir des vecteurs très similaires, rendant l'identification moins précise. C'est comme demander à quelqu'un qui n'a jamais quitté l'Europe de distinguer des villes marocaines sur photo — c'est difficile.

---

## Cellule 9 — Application Streamlit

> **Objectif final du hackathon.**  
> Cette cellule génère le fichier `app.py` et vous indique comment le lancer.

```
  ARCHITECTURE DE L'APP
  ──────────────────────────────────────────────────────────

   Navigateur web
       |
       |  Upload image
       v
  +--------------------+
  |   Streamlit UI     |   <- app.py
  +--------------------+
       |         |
       |         |  Appel direct aux fonctions
       |         |  du notebook (clip_model,
       |         |  collection, generate_response)
       v         v
  +--------+  +----------+
  |  CLIP  |  | ChromaDB |  <- retrieval
  +--------+  +----------+
                   |
                   v
             +----------+
             |  Ollama  |  <- génération
             +----------+
                   |
                   v
          Fiche historique
          affichée dans le
          navigateur
  ──────────────────────────────────────────────────────────
```

**Lancement :**
```bash
# Dans le terminal, avec le venv activé :
streamlit run app.py
```


In [44]:
# ============================================================
#  CELLULE 9 : Génération de app.py
# ============================================================

app_content = [
    "import streamlit as st",
    "import numpy as np",
    "from pathlib import Path",
    "from PIL import Image",
    "import requests",
    "import chromadb",
    "from sentence_transformers import SentenceTransformer",
    "",
    "OLLAMA_URL = 'http://localhost:11434'",
    "",
    "def detecter_modele_ollama(url):",
    "    try:",
    "        r = requests.get(f'{url}/api/tags', timeout=5)",
    "        r.raise_for_status()",
    "        modeles = r.json().get('models', [])",
    "        return modeles[0]['name'] if modeles else None",
    "    except Exception:",
    "        return None",
    "",
    "MODELE_DETECTE = detecter_modele_ollama(OLLAMA_URL)",
    "",
    "CONFIG = {",
    "    'CHROMA_PATH'     : Path('chroma_db'),",
    "    'COLLECTION_NAME' : 'lieux_touristiques_maroc',",
    "    'CLIP_MODEL'      : 'clip-ViT-B-32',",
    "    'OLLAMA_URL'      : OLLAMA_URL,",
    "    'LLM_MODEL'       : MODELE_DETECTE or 'llama3',",
    "    'N_RESULTS'       : 3,",
    "}",
    "",
    "SYSTEM_PROMPT = (",
    "    'Tu es un guide touristique expert specialise dans le patrimoine marocain. '",
    "    'Tu reponds toujours en francais de maniere elegante et pedagogique. '",
    "    'Structure : presentation -> histoire -> anecdote -> conseil visite.'",
    ")",
    "",
    "@st.cache_resource",
    "def load_clip():",
    "    return SentenceTransformer(CONFIG['CLIP_MODEL'])",
    "",
    "@st.cache_resource",
    "def load_collection():",
    "    client = chromadb.PersistentClient(path=str(CONFIG['CHROMA_PATH']))",
    "    return client.get_or_create_collection(",
    "        name=CONFIG['COLLECTION_NAME'],",
    "        metadata={'hnsw:space': 'cosine'},",
    "    )",
    "",
    "def rechercher_lieu(image, clip_model, collection, n_results=3):",
    "    embedding = clip_model.encode(image).tolist()",
    "    raw = collection.query(",
    "        query_embeddings=[embedding],",
    "        n_results=n_results,",
    "        include=['metadatas', 'documents', 'distances'],",
    "    )",
    "    results = []",
    "    for rang, (meta, dist) in enumerate(",
    "        zip(raw['metadatas'][0], raw['distances'][0]), start=1",
    "    ):",
    "        results.append({",
    "            'rang'    : rang,",
    "            'nom'     : meta.get('nom', 'Inconnu'),",
    "            'ville'   : meta.get('ville', 'N/A'),",
    "            'histoire': meta.get('histoire', ''),",
    "            'score'   : round(1 - dist, 4),",
    "        })",
    "    return results",
    "",
    "def generate_response(context, question):",
    "    lignes = [",
    "        '=== LIEU IDENTIFIE ===',",
    "        'Nom    : ' + context['nom'],",
    "        'Ville  : ' + context['ville'],",
    "        'Histoire :',",
    "        context['histoire'],",
    "        '====================',",
    "        '',",
    "        'QUESTION : ' + question,",
    "    ]",
    "    prompt = chr(10).join(lignes)",
    "    payload = {",
    "        'model'   : CONFIG['LLM_MODEL'],",
    "        'messages': [",
    "            {'role': 'system', 'content': SYSTEM_PROMPT},",
    "            {'role': 'user',   'content': prompt},",
    "        ],",
    "        'stream'  : False,",
    "        'options' : {'temperature': 0.7, 'num_predict': 512},",
    "    }",
    "    try:",
    "        resp = requests.post(",
    "            CONFIG['OLLAMA_URL'] + '/api/chat', json=payload, timeout=120",
    "        )",
    "        resp.raise_for_status()",
    "        return resp.json()['message']['content']",
    "    except requests.exceptions.ConnectionError:",
    "        return 'Erreur : Ollama inaccessible. Lancez ollama serve.'",
    "    except requests.exceptions.HTTPError as e:",
    "        if '404' in str(e):",
    "            return 'Modele introuvable. Lancez : ollama pull ' + CONFIG['LLM_MODEL']",
    "        return 'Erreur HTTP : ' + str(e)",
    "    except Exception as e:",
    "        return 'Erreur : ' + str(e)",
    "",
    "def main():",
    "    st.set_page_config(",
    "        page_title='Visual RAG — Tourisme Marocain',",
    "        page_icon='🕌',",
    "        layout='wide',",
    "    )",
    "    st.title('🕌 Visual RAG — Lieux touristiques marocains')",
    "    st.markdown('Uploadez une photo et obtenez une fiche historique generee par IA.')",
    "",
    "    if MODELE_DETECTE is None:",
    "        st.error('Ollama inaccessible. Lancez ollama serve puis ollama pull phi3')",
    "    else:",
    "        st.success('Ollama connecte — modele : ' + MODELE_DETECTE)",
    "",
    "    st.divider()",
    "",
    "    with st.spinner('Chargement CLIP...'):",
    "        clip_model = load_clip()",
    "    with st.spinner('Connexion ChromaDB...'):",
    "        collection = load_collection()",
    "",
    "    n_docs = collection.count()",
    "    if n_docs == 0:",
    "        st.warning('Base vide. Executez la Cellule 4 pour indexer les images.')",
    "        st.stop()",
    "    st.info(f'Base : {n_docs} images indexees')",
    "",
    "    col1, col2 = st.columns([1, 1], gap='large')",
    "",
    "    with col1:",
    "        st.subheader('Image a identifier')",
    "        uploaded = st.file_uploader('Photo (JPG/PNG)', type=['jpg','jpeg','png'],",
    "                                     label_visibility='collapsed')",
    "        question = st.text_area('Question', value='Parle-moi de ce lieu et donne 3 conseils.', height=80)",
    "        n_results = st.slider('Candidats', 1, 5, 3)",
    "        go = st.button('Identifier', type='primary', use_container_width=True,",
    "                        disabled=(MODELE_DETECTE is None))",
    "        if uploaded:",
    "            st.image(uploaded, use_container_width=True)",
    "",
    "    with col2:",
    "        st.subheader('Resultats')",
    "        if go and uploaded:",
    "            image = Image.open(uploaded).convert('RGB')",
    "            with st.spinner('Recherche...'):",
    "                resultats = rechercher_lieu(image, clip_model, collection, n_results)",
    "            if not resultats:",
    "                st.error('Aucun resultat.')",
    "                st.stop()",
    "            for r in resultats:",
    "                st.progress(int(r['score']*100),",
    "                            text=f\"#{r['rang']} {r['nom']} ({r['ville']}) — {r['score']:.4f}\")",
    "            st.divider()",
    "            lieu = resultats[0]",
    "            st.markdown(f\"**Lieu : {lieu['nom']} ({lieu['ville']})**\")",
    "            st.caption(f\"Confiance : {lieu['score']:.2%}\")",
    "            with st.spinner('Generation...'):",
    "                reponse = generate_response(lieu, question)",
    "            st.markdown(reponse)",
    "        elif go:",
    "            st.warning('Uploadez une image.')",
    "        else:",
    "            st.info('Uploadez une image et cliquez sur Identifier.')",
    "",
    "    with st.sidebar:",
    "        st.header('Configuration')",
    "        st.json({",
    "            'CLIP'  : CONFIG['CLIP_MODEL'],",
    "            'LLM'   : CONFIG['LLM_MODEL'],",
    "            'Ollama': CONFIG['OLLAMA_URL'],",
    "            'Images': n_docs,",
    "        })",
    "        st.caption('Hackathon Visual RAG 🇲🇦')",
    "",
    "if __name__ == '__main__':",
    "    main()",
]

with open("app.py", "w", encoding="utf-8") as f:
    f.write("\n".join(app_content))

print("app.py généré sans erreurs de syntaxe !")
print("Lancez : streamlit run app.py")

app.py généré sans erreurs de syntaxe !
Lancez : streamlit run app.py
